# BraTS 2024 - Missing Modality Synthesis

Trains a U-Net generator that synthesizes one missing MRI modality from the other three (T1, T1CE, T2, FLAIR).
Loss: masked **L1 + (1 - SSIM)** computed inside the brain only (background is exactly 0 after z-scoring).
Metrics: SSIM, PSNR, MAE inside the brain mask.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

repo_src = PROJECT_ROOT / "src"
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

import torch
import numpy as np
import matplotlib.pyplot as plt

from brain_mri.data.dataset import create_dataloader, MODALITIES
from brain_mri.models.unet_generator import Unet_Generator
from brain_mri.models.losses import MaskedL1SSIMLoss
from brain_mri.training import SynthesisTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device}")
if device.type == "cuda":
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")

## 2. DataLoaders

In [ ]:
DATA_DIR = Path(r"E:bronikowski\data")
TARGET = "T1CE"  # modality to synthesize; one of MODALITIES

BATCH_SIZE = 32
NUM_WORKERS = 2  # Set to 0 if multiprocessing errors occur on Windows
PIN_MEMORY = torch.cuda.is_available()

def make_loader(split, shuffle):
    return create_dataloader(
        split_dir=DATA_DIR / "processed_no_leaks" / split,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        target_modality=TARGET,
    )

train_loader = make_loader("train", True)
val_loader = make_loader("validate", False)
test_loader = make_loader("test", False)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)} slices")

checkpoint_dir = PROJECT_ROOT / "checkpoints"
EPOCHS = 30

## 3. Model, Loss, Trainer

In [ ]:
model = Unet_Generator(in_channels=3, out_channels=1, base_features=64)

criterion = MaskedL1SSIMLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

trainer = SynthesisTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=checkpoint_dir,
    scheduler=scheduler,
    checkpoint_name=f"best_generator_{TARGET.lower()}.pt",
)

## 4. Training

In [ ]:
history = trainer.fit(epochs=EPOCHS)

## 5. Learning Curves

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", linewidth=2)
plt.plot(history["val_loss"], label="Val Loss", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss During Training")
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history["val_ssim"], label="Val SSIM", color="green", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("SSIM"); plt.title("Validation SSIM (brain mask)")
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.savefig(checkpoint_dir / f"learning_curves_generator_{TARGET.lower()}.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Test Set Evaluation

Loads the best checkpoint (selected on validation SSIM) and evaluates it once on the held-out test split.

In [ ]:
checkpoint = torch.load(checkpoint_dir / f"best_generator_{TARGET.lower()}.pt", map_location=device, weights_only=True)
model = Unet_Generator(in_channels=3, out_channels=1, base_features=64).to(device, memory_format=torch.channels_last)
model.load_state_dict(checkpoint["model_state_dict"])
trainer.model = model
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val SSIM: {checkpoint['best_ssim']:.4f})")

test_loss, test_metrics = trainer.validate_epoch(test_loader)
print(f"Test loss: {test_loss:.4f}")
print(f"Test SSIM: {test_metrics['ssim']:.4f} | PSNR: {test_metrics['psnr']:.2f} dB | MAE: {test_metrics['mae']:.4f}")

## 7. Example Syntheses

Random test slices: inputs, ground truth, synthesized image, and absolute error (brain only).

In [ ]:
N_EXAMPLES = 4
input_names = [m for m in MODALITIES if m != TARGET]

rng = np.random.default_rng(42)
indices = rng.choice(len(test_loader.dataset), size=N_EXAMPLES, replace=False)

model.eval()
fig, axes = plt.subplots(N_EXAMPLES, 6, figsize=(15, 2.6 * N_EXAMPLES))
for row, idx in enumerate(indices):
    inputs, target, mask = test_loader.dataset[int(idx)]
    with torch.no_grad():
        pred = model(inputs[None].to(device, memory_format=torch.channels_last)).float().cpu()[0]
    err = (pred - target).abs() * mask

    panels = [(inputs[i], input_names[i]) for i in range(3)] + [
        (target[0], f"{TARGET} (real)"), (pred[0], f"{TARGET} (synth)"), (err[0], "|error|")
    ]
    vmax = target.max().item()
    for col, (img, title) in enumerate(panels):
        ax = axes[row, col]
        if title == "|error|":
            ax.imshow(img, cmap="hot", vmin=0, vmax=2)
        else:
            ax.imshow(img, cmap="gray", vmin=-1, vmax=vmax)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.savefig(checkpoint_dir / f"test_examples_generator_{TARGET.lower()}.png", dpi=150, bbox_inches="tight")
plt.show()